# Fine-tune Mistral-7B with QLoRA for Review Summarization

This notebook fine-tunes Mistral-7B-Instruct using QLoRA (4-bit quantization + LoRA adapters) on 200 product review summary pairs.

**Requirements:** Run on Google Colab with T4 GPU (free tier).

**Steps:**
1. Upload `summary_training_pairs.jsonl` from `data/processed/`
2. Install dependencies
3. Load and format training data
4. Load Mistral-7B in 4-bit
5. Apply LoRA adapters
6. Train for 3 epochs
7. Save adapter weights
8. Test generation
9. Download adapter weights

In [ ]:
# Step 1: Upload training data
from google.colab import files
uploaded = files.upload()  # Upload summary_training_pairs.jsonl

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate trl

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 3: Load and format training data
pairs = []
with open('summary_training_pairs.jsonl') as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} training pairs")

# Format as chat messages for Mistral-Instruct
def format_prompt(pair):
    return (
        f"<s>[INST] You are an expert product analyst. "
        f"Given the following customer reviews, write a structured executive summary.\n\n"
        f"{pair['input'][:3000]}\n[/INST]\n"
        f"{pair['summary']}</s>"
    )

formatted = [{'text': format_prompt(p)} for p in pairs]
dataset = Dataset.from_list(formatted)

# Train/val split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds = split['test']
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"\nSample:\n{formatted[0]['text'][:500]}...")

In [ ]:
# Step 4: Load Mistral-7B in 4-bit quantization
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

In [ ]:
# Step 5: Apply LoRA adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% of total parameters are trainable

In [ ]:
# Step 6: Train for 3 epochs
training_args = TrainingArguments(
    output_dir="./mistral-qlora-reviews",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=2048,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Step 7: Save adapter weights
ADAPTER_DIR = "./mistral-qlora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

# Check adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f"Adapter size: {total_size:.1f} MB")

In [ ]:
# Step 8: Test generation — compare base vs fine-tuned
test_input = pairs[-1]['input'][:2000]  # Use last pair as test

prompt = (
    f"<s>[INST] You are an expert product analyst. "
    f"Given the following customer reviews, write a structured executive summary.\n\n"
    f"{test_input}\n[/INST]\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the response (after [/INST])
response = generated.split('[/INST]')[-1].strip()
print("=== Fine-tuned Mistral Output ===")
print(response)

In [ ]:
# Step 9: Download adapter weights
!zip -r mistral-qlora-adapter.zip mistral-qlora-adapter/
files.download('mistral-qlora-adapter.zip')
print("\nDownload the zip and extract to models/mistral-qlora/ in your project")

In [ ]:
# Step 10: Also save training loss for documentation
history = trainer.state.log_history
print("\n=== Training History ===")
for entry in history:
    if 'loss' in entry:
        print(f"Step {entry.get('step', '?')}: loss={entry['loss']:.4f}")
    if 'eval_loss' in entry:
        print(f"  eval_loss={entry['eval_loss']:.4f}")